# Adım 4 — Baseline eğitim (YOLOv8s, imgsz 640)

**Kaggle ayarları (sağ panel):** Accelerator = **GPU T4** (P100 değil), Internet = **açık**,
"Add Input" ile `rdd2022_yolo` dataset'i eklenir.

**Hangi hücre ne zaman?**
1. **İlk oturum (deneme):** `RUN_TRIAL = True` → hücreleri sırayla çalıştır. Deneme koşusu (a) 1. epoch bitince
   (log'da `2/3` başlayınca) **Interrupt** ile bilerek durdurulur. Sonra `RESUME = True`, `RESUME_RUN = "deneme"`
   yapılıp (b) hücresi çalıştırılır → kaldığı epoch'tan devam ettiğini gör. Epoch süresini not al.
2. **Asıl eğitim:** yeni oturumda `RUN_TRIAL = False`, `RUN_MAIN = True` → **Save Version → Save & Run All (Commit)**.
   Tarayıcı kapansa da çalışır; çıktılar (`/kaggle/working/runs`) notebook'un output'u olarak saklanır.
3. **12 saat dolarsa:** önceki versiyonun output'unu "Add Input" ile ekle, `RESUME = True`, `RESUME_RUN = "baseline_v8s"`,
   `PREV_OUTPUT` = o output'taki `runs/baseline_v8s` yolu; `RUN_MAIN = False` → Save & Run All.

## 1. GPU ve kurulum

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q ultralytics==8.4.182
import torch, ultralytics
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| CUDA", torch.cuda.is_available())

## 2. Veriyi bul ve kontrol et
Kaggle zip'i açarken klasör adını kendisi koyar; bu yüzden `data.yaml` aranarak bulunur.
Sayılar Adım 3 ile tutmalı: train 21.211 / val 3.838 / test 3.838.

In [ ]:
from pathlib import Path

EXPECTED = {"train": 21211, "val": 3838, "test": 3838}

candidates = [p.parent for p in Path("/kaggle/input").glob("**/data.yaml") if (p.parent / "train.txt").exists()]
assert len(candidates) == 1, f"Tek veri klasörü bekleniyordu, bulunan: {candidates}"
DATA_DIR = candidates[0]
print("Veri klasörü:", DATA_DIR)

for part, n in EXPECTED.items():
    lines = (DATA_DIR / f"{part}.txt").read_text(encoding="utf-8").splitlines()
    n_img = sum(1 for _ in (DATA_DIR / "images" / part).iterdir())
    n_lbl = sum(1 for _ in (DATA_DIR / "labels" / part).iterdir())
    first = DATA_DIR / lines[0]
    print(f"{part}: liste {len(lines)}, görüntü {n_img}, etiket {n_lbl}, ilk satır var mı: {first.exists()}")
    assert len(lines) == n_img == n_lbl == n and first.exists()

## 3. data.yaml'ı /kaggle/working'e yaz
`/kaggle/input` salt okunur. Yeni yaml `path` ile input klasörünü gösterir; listelerdeki `./images/...` satırları
Ultralytics tarafından listenin bulunduğu klasöre göre çözülür. (Etiket önbelleği `.cache` input'a yazılamaz,
Ultralytics bunu uyarıyla geçer; sorun değil.)
**Resume'dan önce de çalıştırılmalı:** checkpoint bu yaml'ın yolunu saklıyor.

In [ ]:
DATA_YAML = Path("/kaggle/working/data.yaml")
DATA_YAML.write_text(
    f"path: {DATA_DIR}\n"
    "train: train.txt\nval: val.txt\ntest: test.txt\n\n"
    "names:\n  0: D00\n  1: D10\n  2: D20\n  3: D40\n",
    encoding="utf-8",
)
print(DATA_YAML.read_text())

## 4. Ortak ayarlar ve hangi hücrelerin çalışacağı
- `patience=10`: val mAP 10 epoch iyileşmezse durur (early stopping).
- `save_period=5`: her 5 epoch'ta ayrıca `epochN.pt` kaydı (`last.pt` zaten her epoch yazılır).
- `project` tam yol: göreli verilince Ultralytics başına `runs/detect` ekliyor (Adım 3'te görüldü).

In [ ]:
from ultralytics import YOLO
import shutil

RUN_TRIAL = True    # (a) deneme koşusu
RESUME = False      # (b) last.pt'den devam
RUN_MAIN = False    # (c) asıl eğitim

RESUME_RUN = "deneme"   # devam ettirilecek koşu: "deneme" ya da "baseline_v8s"
PREV_OUTPUT = None      # yeni oturumda: önceki output'taki koşu klasörü, ör. "/kaggle/input/<notebook>/runs/baseline_v8s"

RUNS = Path("/kaggle/working/runs")
COMMON = dict(
    data=str(DATA_YAML), imgsz=640, seed=42, cache=False, batch=16, workers=4,
    patience=10, save_period=5, project=str(RUNS), device=0,
)

## (a) Deneme koşusu — 3 epoch
İlk oturumda çalıştırılır. Amaç: hattın Kaggle'da çalıştığını görmek, epoch süresini ölçmek (kota hesabı) ve
1. epoch'tan sonra **bilerek durdurup** (b) ile devam ettirmeyi denemek.

In [ ]:
if RUN_TRIAL:
    model = YOLO("yolov8s.pt")  # COCO ön-eğitimli ağırlıklar (transfer learning)
    model.train(**COMMON, epochs=3, name="deneme")
    print("Çıktı klasörü:", model.trainer.save_dir)

## (b) last.pt'den devam (resume)
- **Aynı oturumda** (deneme durdurulduysa): `RESUME = True`, `RESUME_RUN = "deneme"`, `PREV_OUTPUT = None`.
- **Yeni oturumda** (12 saat dolduysa): önceki output'u input olarak ekle, `PREV_OUTPUT`'u ver. Klasör aynı yola
  (`/kaggle/working/runs/<ad>`) kopyalanır çünkü checkpoint mutlak yolları saklıyor. Bölüm 2-3 önce çalışmış olmalı.

In [ ]:
if RESUME:
    run_dir = RUNS / RESUME_RUN
    if PREV_OUTPUT and not run_dir.exists():
        shutil.copytree(PREV_OUTPUT, run_dir)
    last = run_dir / "weights" / "last.pt"
    assert last.exists(), f"{last} yok"
    model = YOLO(str(last))
    model.train(resume=True)  # ayarlar checkpoint'ten okunur
    print("Çıktı klasörü:", model.trainer.save_dir)

## (c) Asıl eğitim — YOLOv8s, 640, 50 epoch
Deneme + resume başarılı olduktan sonra, yeni oturumda **Save & Run All (Commit)** ile çalıştırılır.

In [ ]:
if RUN_MAIN:
    model = YOLO("yolov8s.pt")
    model.train(**COMMON, epochs=50, name="baseline_v8s")
    print("Çıktı klasörü:", model.trainer.save_dir)

## Sonuç özeti

In [ ]:
import pandas as pd

for run_dir in sorted(RUNS.glob("*")):
    csv = run_dir / "results.csv"
    if csv.exists():
        df = pd.read_csv(csv)
        df.columns = df.columns.str.strip()
        print(f"== {run_dir.name}: {len(df)} epoch, en iyi val mAP50 {df['metrics/mAP50(B)'].max():.3f}")
        print(df[["epoch", "train/box_loss", "train/cls_loss", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]].tail(5).to_string(index=False))
        print("best.pt:", run_dir / "weights" / "best.pt", "\n")